# Stage 2: Brown Shipley Transformation

**Purpose**: Transform Stage 1 raw data into standardized `individual_dfm_consolidated` format

**Replicates Excel Workbook's "Edited" Sheet**:
- Policy mapping (Account Number → Policyholder_Number via Mappings sheet)
- Security mapping (composite key: identifier-currency lookup in Mappings)
- Include/Remove flagging based on IH Report + policy conditions
- `Movt`-based tolerance check flags for validation (98-102)
- FX conversion to GBP using Check sheet rates
- Decision traceability

**Input**: `stage1_brown_shipley_positions_raw`, `stage1_brown_shipley_cash_raw`

**Output**: `individual_dfm_consolidated` (row-for-row with Stage 1, enriched)

**Row Count Expectation**: Same as Stage 1 raw (e.g., 220 in → 220 out, with flags)

In [23]:
# ========== Establish Workspace Context ==========
# CRITICAL: Required when notebook is in a subfolder
# This ensures Fabric's lakehouse context is properly initialized
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")
    print("[ACTION] Ensure lakehouse is attached in Fabric notebook UI")

# ========== Parameters ==========
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by orchestrator in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

dfm_id = "brown_shipley"
dfm_name = "Brown Shipley"
profile_id = "brown_shipley_default"

print(f"[STAGE 2] Starting Brown Shipley transformation")
print(f"  Period: {period}")
print(f"  Run ID: {run_id}")
print(f"  DFM: {dfm_name}")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 26, Finished, Available, Finished, False)

[INFO] Workspace context initialized
[STAGE 2] Starting Brown Shipley transformation
  Period: 2026-03
  Run ID: manual_test_run
  DFM: Brown Shipley


In [24]:
# ========== Imports ==========
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
import json

print("[STAGE 2] Imports complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 27, Finished, Available, Finished, False)

[STAGE 2] Imports complete


## Step 1: Read Stage 1 Raw Tables

Read positions and cash data from Stage 1 raw tables (created by v2 ingestion).

In [25]:
# ========== Read Stage 1 Raw Data ==========
print("\n[STEP 1] Reading Stage 1 raw tables...")
print("=" * 70)

def table_exists(table_name):
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False

def dedupe_stage1(df, label):
    # Deduplicate by ingestion provenance keys to prevent double-loaded rows.
    dedupe_keys_preferred = ["period", "run_id", "source_file", "source_row_id"]
    dedupe_keys = [c for c in dedupe_keys_preferred if c in df.columns]

    raw_count = df.count()
    print(f"  {label} (raw): {raw_count} rows")

    if "source_row_id" in dedupe_keys and "source_file" in dedupe_keys:
        deduped_df = df.dropDuplicates(dedupe_keys)
        deduped_count = deduped_df.count()
        removed = raw_count - deduped_count
        print(f"  {label} (deduped): {deduped_count} rows using keys {dedupe_keys}")
        if removed > 0:
            print(f"  [WARNING] {label} duplicate rows removed in Stage 2: {removed}")
        return deduped_df, deduped_count

    print(f"  [WARNING] {label}: source provenance keys missing; no dedupe applied")
    return df, raw_count

# Read positions (securities)
positions_table = "stage1_brown_shipley_positions_raw"
if not table_exists(positions_table):
    print(f"[ERROR] Table {positions_table} not found. Run nb_ingest_brown_shipley_v2 first.")
    mssparkutils.notebook.exit("FAILED")

positions_raw = (
    spark.table(positions_table)
    .filter((F.col("period") == period) & (F.col("run_id") == run_id))
)
positions, positions_count = dedupe_stage1(positions_raw, "Positions (securities)")

# Read cash
cash_table = "stage1_brown_shipley_cash_raw"
cash_count = 0
if table_exists(cash_table):
    cash_raw = (
        spark.table(cash_table)
        .filter((F.col("period") == period) & (F.col("run_id") == run_id))
    )
    cash, cash_count = dedupe_stage1(cash_raw, "Cash")
else:
    print("  Cash: table not found (will use positions only)")
    cash = None

if positions_count == 0:
    print("[ERROR] No data found for this period/run_id. Check Stage 1 ingestion.")
    mssparkutils.notebook.exit("NO_DATA")

print("\n[STEP 1] Stage 1 data loaded successfully")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 28, Finished, Available, Finished, False)


[STEP 1] Reading Stage 1 raw tables...
  Positions (securities) (raw): 30548 rows
  Positions (securities) (deduped): 15274 rows using keys ['period', 'run_id', 'source_file', 'source_row_id']
  [WARNING] Positions (securities) duplicate rows removed in Stage 2: 15274
  Cash (raw): 1592 rows
  Cash (deduped): 796 rows using keys ['period', 'run_id', 'source_file', 'source_row_id']
  [WARNING] Cash duplicate rows removed in Stage 2: 796

[STEP 1] Stage 1 data loaded successfully


## Step 2: Combine Positions + Cash

Row-preserving append of positions and cash data (no join). This keeps one output row per input source row.

In [26]:
# ========== Combine Positions and Cash ==========
print("\n[STEP 2] Combining positions and cash...")
print("=" * 70)

# Detect optional columns once.
pos_has_movt = "movt" in positions.columns
cash_has_movt = cash is not None and ("movt" in cash.columns)

# Detect source columns that map to workbook value fields.
def first_existing(df, candidates):
    cols = {c.lower().strip(): c for c in df.columns}
    for candidate in candidates:
        if candidate in cols:
            return cols[candidate]
    return None

# L source (Original Data - Non Cash!I): value to convert into Bid_Value_in_GBP.
pos_value_col = first_existing(
    positions,
    [
        "cash_evaluation",
        "cash evaluation",
        "market_value",
        "market value",
        "value_in_market",
        "value in market",
        "bid_value_local",
        "valuation",
        "value"
    ]
)

# O source fallback (if unit price already exists in source).
pos_price_col = first_existing(
    positions,
    [
        "loc_bid_price",
        "local_bid_price",
        "bid_price",
        "bid price",
        "price"
    ]
)

print(f"  Value source column (for non-cash L): {pos_value_col if pos_value_col else 'not found; will fallback in Step 7'}")
print(f"  Price source column (for O fallback): {pos_price_col if pos_price_col else 'not found; will derive in Step 7'}")

# Normalize positions to Stage 2 working schema (row-preserving).
positions_normalized = positions.select(
    "period", "run_id", "dfm_id", "source_file", "source_row_id",
    F.col("client_id").alias("client_id"),
    F.col("value_date").alias("value_date"),
    F.col("isin_code").alias("isin"),
    F.col("sedol_code").alias("sedol"),
    F.col("description_of_security").alias("instrument_name"),
    F.col("type_of_financial_instrument").alias("instrument_type"),
    F.col("balance").alias("balance_local"),
    F.col("accrued_interest").alias("accrued_interest_local"),
    F.col("currency_code").alias("position_currency"),
    (F.col(pos_value_col).cast("string") if pos_value_col else F.lit(None).cast("string")).alias("position_value_local_raw"),
    (F.col(pos_price_col).cast("string") if pos_price_col else F.lit(None).cast("string")).alias("position_local_bid_price_raw"),
    F.lit(None).cast("string").alias("cash_balance_local"),
    F.lit(None).cast("string").alias("cash_currency"),
    (F.col("movt") if pos_has_movt else F.lit(None).cast("string")).alias("movt_raw"),
    F.lit("POSITION").alias("source_record_type")
)

if cash is not None and cash.count() > 0:
    # Normalize cash to same schema (row-preserving).
    cash_normalized = cash.select(
        "period", "run_id", "dfm_id", "source_file", "source_row_id",
        F.col("client_id").alias("client_id"),
        F.col("value_date").alias("value_date"),
        F.lit(None).cast("string").alias("isin"),
        F.lit(None).cast("string").alias("sedol"),
        F.lit("Cash").alias("instrument_name"),
        F.lit("Cash").alias("instrument_type"),
        F.lit(None).cast("string").alias("balance_local"),
        F.lit(None).cast("string").alias("accrued_interest_local"),
        F.col("account_currency").alias("position_currency"),
        F.lit(None).cast("string").alias("position_value_local_raw"),
        F.lit(None).cast("string").alias("position_local_bid_price_raw"),
        F.col("accounting_balance").alias("cash_balance_local"),
        F.col("account_currency").alias("cash_currency"),
        (F.col("movt") if cash_has_movt else F.lit(None).cast("string")).alias("movt_raw"),
        F.lit("CASH").alias("source_record_type")
    )

    combined = positions_normalized.unionByName(cash_normalized)
    print(f"  Movt column source: positions={pos_has_movt}, cash={cash_has_movt}")
else:
    combined = positions_normalized
    print(f"  Movt column source: positions={pos_has_movt}, cash=False")

combined_count = combined.count()
expected_count = positions_count + (cash_count if cash is not None else 0)

print(f"  Combined rows: {combined_count}")
print(f"  Expected: {expected_count} (positions + cash, row-preserving)")

if combined_count != expected_count:
    print(f"  [WARNING] Row count mismatch. Expected {expected_count}, got {combined_count}")

print("\n[STEP 2] Positions and cash combined successfully")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 29, Finished, Available, Finished, False)


[STEP 2] Combining positions and cash...
  Value source column (for non-cash L): cash_evaluation
  Price source column (for O fallback): not found; will derive in Step 7
  Movt column source: positions=False, cash=False
  Combined rows: 16070
  Expected: 16070 (positions + cash, row-preserving)

[STEP 2] Positions and cash combined successfully


In [27]:
# ========== Step 2A: Add Workbook Value/Price Source Columns ==========
# Ensures downstream Step 7 can mirror workbook L/N/O formulas.
print("\n[STEP 2A] Adding workbook source columns (value + unit price)...")

if "position_value_local_raw" not in combined.columns:
    # If Stage 2 base schema omitted this field, backfill with best available source.
    fallback_value_col = "balance_local" if "balance_local" in combined.columns else None
    combined = combined.withColumn(
        "position_value_local_raw",
        F.col(fallback_value_col).cast("string") if fallback_value_col else F.lit(None).cast("string")
    )

if "position_local_bid_price_raw" not in combined.columns:
    combined = combined.withColumn("position_local_bid_price_raw", F.lit(None).cast("string"))

print("  position_value_local_raw available")
print("  position_local_bid_price_raw available")
print("[STEP 2A] Complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 30, Finished, Available, Finished, False)


[STEP 2A] Adding workbook source columns (value + unit price)...
  position_value_local_raw available
  position_local_bid_price_raw available
[STEP 2A] Complete


## Step 3: Load Reference Data

Load the three reference datasets that drive the workbook logic.

- **Mappings**: Policy and security identifier mappings (DFM-scoped)
- **IH Report**: Policy inclusion authority
- **FX Rates**: Currency conversion to GBP

Reference paths (DFM-scoped first, no global fallback):
- Mappings: `Files/config/{dfm_id}/mappings.csv` or `Files/config/{dfm_id}/policy_mapping.csv`
- IH Report: `Files/config/{dfm_id}/ih_report_mapping.csv` → shared → table
- FX Rates: table `fx_rates` (preferred) → `Files/config/fx_rates.csv`

In [28]:
# ========== Load Reference Data ==========
print("\n[STEP 3] Loading reference data...")
print("=" * 70)

import pandas as pd

# Helper: try loading first available Excel file from candidate paths using pandas.
def try_read_excel(paths, sheet=None):
    last_error = None
    for p in paths:
        try:
            # Read Excel using pandas (Fabric-compatible)
            # Convert Fabric path to absolute path for pandas
            file_path = f"/lakehouse/default/{p}"
            pdf = pd.read_excel(file_path, sheet_name=sheet if sheet else 0)
            # Convert pandas DataFrame to Spark DataFrame
            df = spark.createDataFrame(pdf)
            return df, p, None
        except Exception as e:
            last_error = e
    return None, None, last_error

# Helper: map lower-cased columns to original names.
def colmap(df):
    return {c.lower().strip(): c for c in df.columns}

# -------------------------------------------------
# Mappings (DFM-scoped, authoritative source)
# -------------------------------------------------
mappings_paths = [
    f"Files/config/{dfm_id}/mappings.xlsx",
]

mappings_raw, mappings_source, mappings_err = try_read_excel(mappings_paths)
if mappings_raw is not None:
    print(f"  Mappings loaded: {mappings_source}")
    print(f"  Columns: {mappings_raw.columns}")
else:
    print(f"  [WARNING] Could not load mappings from: {mappings_paths}")
    print(f"  Last error: {mappings_err}")
    mappings_raw = None

# -------------------------------------------------
# IH Report (DFM-specific first, Excel format)
# -------------------------------------------------
ih_paths = [
    f"Files/config/{dfm_id}/ih_report_mapping.xlsx",
]
ih_raw, ih_source, ih_err = try_read_excel(ih_paths)
if ih_raw is None:
    try:
        if table_exists("ih_report_mapping"):
            ih_raw = spark.table("ih_report_mapping")
            ih_source = "table:ih_report_mapping"
    except Exception:
        ih_raw = None

if ih_raw is not None:
    print(f"  IH Report loaded: {ih_source}")
    print(f"  Columns: {ih_raw.columns}")
else:
    print(f"  [WARNING] Could not load IH Report from: {ih_paths}")
    if ih_err is not None:
        print(f"  Last error: {ih_err}")
    ih_raw = None

# -------------------------------------------------
# FX Rates (prefer table, fallback to file)
# -------------------------------------------------
fx_rates = None
try:
    fx_rates = (
        spark.table("fx_rates")
        .filter(F.col("period") == period)
        .select(
            F.upper(F.trim(F.col("from_currency"))).alias("currency"),
            F.col("rate").cast("double").alias("fx_rate")
        )
        .dropDuplicates(["currency"])
    )
    fx_count = fx_rates.count()
    print(f"  FX rates (table): {fx_count} currencies for period={period}")
except Exception as e:
    print(f"  [WARNING] Could not load fx_rates table: {e}")
    try:
        fx_rates = (
            spark.read.option("header", True).csv("Files/config/fx_rates.csv")
            .select(
                F.upper(F.trim(F.col("currency"))).alias("currency"),
                F.col("rate_to_gbp").cast("double").alias("fx_rate")
            )
        )
        fx_count = fx_rates.count()
        print(f"  FX rates (file): {fx_count} currencies")
    except Exception as e2:
        print(f"  [WARNING] Could not load FX rates file: {e2}")
        fx_rates = None

print("\n[STEP 3] Reference data loaded")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 31, Finished, Available, Finished, False)


[STEP 3] Loading reference data...
  Mappings loaded: Files/config/brown_shipley/mappings.xlsx
  Columns: ['UBS Ref', 'SL PolNo', 'Unnamed: 2', 'ISIN+CCY', 'final_security_code', 'unique_security_code', 'isin', 'currency', 'sedol', 'SECURITY_NAME']
  IH Report loaded: Files/config/brown_shipley/ih_report_mapping.xlsx
  Columns: ['Policy number', 'Valuation']
  FX rates (table): 30 currencies for period=2026-03

[STEP 3] Reference data loaded


/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Expected bytes, got a 'int' object
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


## Step 4: Policy Mapping

Map `client_id` → `policyholder_number` using policy_mapping reference.

In [29]:
# ========== Apply Policy Mapping ==========
print("\n[STEP 4] Applying policy mapping...")
print("=" * 70)

# Extract policy mapping from raw mappings table
policy_mapping = None
if mappings_raw is not None:
    mcols = colmap(mappings_raw)

    src_col = None
    for candidate in ["ubs ref", "ubs_ref", "dfm_policy_ref", "client_id", "source_policy_ref"]:
        if candidate in mcols:
            src_col = mcols[candidate]
            break

    tgt_col = None
    for candidate in ["sl polno", "sl_polno", "ih_policy_ref", "policyholder_number", "target_policy_ref"]:
        if candidate in mcols:
            tgt_col = mcols[candidate]
            break

    if src_col is None or tgt_col is None:
        print(f"  [WARNING] Mappings table found but policy columns missing.")
        print(f"  Expected source like: {['ubs ref', 'ubs_ref']}; expected target like: {['sl polno', 'sl_polno']}")
    else:
        policy_mapping = (
            mappings_raw
            .select(
                F.trim(F.col(src_col)).alias("client_id_map"),
                F.trim(F.col(tgt_col)).alias("policyholder_number")
            )
            .filter(F.col("client_id_map").isNotNull() & (F.trim(F.col("client_id_map")) != ""))
            .filter(F.col("policyholder_number").isNotNull() & (F.trim(F.col("policyholder_number")) != ""))
            .dropDuplicates(["client_id_map", "policyholder_number"])
        )

        policy_map_count = policy_mapping.count()
        print(f"  Policy mappings: {policy_map_count} rows")

if policy_mapping is not None:
    enriched = combined.join(
        policy_mapping,
        combined["client_id"] == policy_mapping["client_id_map"],
        "left"
    )

    enriched = enriched.withColumn(
        "policy_mapping_applied",
        F.when(F.col("policyholder_number").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )

    enriched = enriched.withColumn(
        "policyholder_number",
        F.coalesce(F.col("policyholder_number"), F.col("client_id"))
    )

    mapped_count = enriched.filter(F.col("policy_mapping_applied") == True).count()
    print(f"  Rows mapped: {mapped_count} / {combined_count}")
    print(f"  Rows using client_id fallback: {combined_count - mapped_count}")
else:
    enriched = combined.withColumn("policyholder_number", F.col("client_id"))
    enriched = enriched.withColumn("policy_mapping_applied", F.lit(False))
    print("  Using client_id as policyholder_number (no mapping table)")

# Enrich IH Report data by policy number
if ih_raw is not None:
    ih_cols = colmap(ih_raw)

    policy_col = None
    for candidate in ["policy_number", "policyholder_number", "policy_ref", "client_id", "policy", "policy number"]:
        if candidate in ih_cols:
            policy_col = ih_cols[candidate]
            break

    status_col = None
    for candidate in ["ih_status", "status", "exclude_flag", "include_exclude", "action"]:
        if candidate in ih_cols:
            status_col = ih_cols[candidate]
            break

    if policy_col is None:
        print(f"  [WARNING] IH Report found but no policy column detected.")
        enriched = enriched.withColumn("ih_lookup_count", F.lit(0))
        enriched = enriched.withColumn("ih_exclude_flag", F.lit(False))
        enriched = enriched.withColumn("ih_status_raw", F.lit(None).cast("string"))
        enriched = enriched.withColumn("ih_policy_exists", F.lit(False))
    else:
        ih_df = ih_raw
        if "dfm_id" in ih_cols:
            ih_df = ih_df.filter(F.lower(F.trim(F.col(ih_cols["dfm_id"]))) == F.lit(dfm_id.lower()))

        ih_policy_rollup = (
            ih_df
            .filter(F.col(policy_col).isNotNull() & (F.trim(F.col(policy_col)) != ""))
            .select(
                F.upper(F.trim(F.col(policy_col))).alias("ih_policy_number"),
                F.col(status_col).alias("ih_status_raw") if status_col is not None else F.lit(None).cast("string").alias("ih_status_raw")
            )
            .groupBy("ih_policy_number")
            .agg(
                F.count(F.lit(1)).alias("ih_lookup_count"),
                F.max(
                    F.when(
                        F.upper(F.trim(F.coalesce(F.col("ih_status_raw"), F.lit("")))) == "EXCLUDE",
                        F.lit(1)
                    ).otherwise(F.lit(0))
                ).alias("ih_exclude_flag_int"),
                F.first(F.col("ih_status_raw"), ignorenulls=True).alias("ih_status_raw")
            )
            .withColumn("ih_exclude_flag", F.col("ih_exclude_flag_int") == 1)
            .drop("ih_exclude_flag_int")
        )

        enriched = enriched.join(
            ih_policy_rollup,
            F.upper(F.trim(enriched["policyholder_number"])) == ih_policy_rollup["ih_policy_number"],
            "left"
        )

        enriched = enriched.withColumn("ih_lookup_count", F.coalesce(F.col("ih_lookup_count"), F.lit(0)))
        enriched = enriched.withColumn("ih_exclude_flag", F.coalesce(F.col("ih_exclude_flag"), F.lit(False)))
        enriched = enriched.withColumn("ih_policy_exists", F.col("ih_lookup_count") > 0)

        ih_match_count = enriched.filter(F.col("ih_policy_exists") == True).count()
        ih_exclude_count = enriched.filter(F.col("ih_exclude_flag") == True).count()

        print(f"  IH policy matches: {ih_match_count}")
        print(f"  IH Exclude flags: {ih_exclude_count}")
else:
    enriched = enriched.withColumn("ih_lookup_count", F.lit(0))
    enriched = enriched.withColumn("ih_exclude_flag", F.lit(False))
    enriched = enriched.withColumn("ih_status_raw", F.lit(None).cast("string"))
    enriched = enriched.withColumn("ih_policy_exists", F.lit(False))
    print("  IH Report not available; all rows will default to Remove in include logic")

print("\n[STEP 4] Policy mapping complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 32, Finished, Available, Finished, False)


[STEP 4] Applying policy mapping...
  Policy mappings: 220 rows
  Rows mapped: 16320 / 16070
  Rows using client_id fallback: -250
  IH policy matches: 9070
  IH Exclude flags: 0

[STEP 4] Policy mapping complete


## Step 5: Security Identifier Resolution

Priority: SEDOL → ISIN → Synthetic cash code (`TPY_CASH_{CURRENCY}`).

In [30]:
# ========== Security Identifier Resolution & Mapping ==========
print("\n[STEP 5] Resolving security identifiers and applying security mapping...")
print("=" * 70)

# Detect cash lines: both sedol and isin are null
enriched = enriched.withColumn(
    "_is_cash_line",
    F.col("sedol").isNull() & F.col("isin").isNull()
)

# Priority-based security identifier selection
# Excel formula: IF(N8937="", P8937, N8937) & "-" & J8937
# Translation: Use ISIN (N), fallback to SEDOL (P), append Currency (J)
# For cash: synthetic CASH_{CURRENCY} code
# For non-cash: ISIN → SEDOL fallback
enriched = enriched.withColumn(
    "derived_security_id",
    F.when(
        F.col("_is_cash_line"),
        F.concat(F.lit("CASH_"), F.upper(F.trim(F.col("position_currency"))))
    ).otherwise(
        F.coalesce(
            F.when((F.col("isin").isNotNull()) & (F.trim(F.col("isin")) != ""), F.upper(F.trim(F.col("isin")))),
            F.when((F.col("sedol").isNotNull()) & (F.trim(F.col("sedol")) != ""), F.upper(F.trim(F.col("sedol"))))
        )
    )
)

derived_count = enriched.filter(F.col("derived_security_id").isNotNull()).count()
print(f"  Derived security IDs: {derived_count} / {enriched.count()}")

# Identify cash lines
cash_line_count = enriched.filter(F.col("_is_cash_line")).count()
print(f"  Cash lines (synthetic codes): {cash_line_count}")

# Apply security mapping using composite key (identifier + currency)
if mappings_raw is not None:
    scols = colmap(mappings_raw)
    
    print("  Detected mapping columns: " + str(list(scols.keys())))

    # First check for composite key column (e.g., "ISIN+CCY" or "isin-ccy")
    composite_col = None
    for candidate in ["isin+ccy", "isin-ccy", "identifier+currency", "security_id+ccy"]:
        if candidate in scols:
            composite_col = scols[candidate]
            print("  Found composite key column: " + composite_col)
            break
    
    identifier_col = None
    currency_col = None
    
    if composite_col is not None:
        # Parse composite key: split on "-" or trailing 3-char currency
        # Example: "NL0010273215-EUR" -> identifier="NL0010273215", currency="EUR"
        print("  Parsing composite key from column: " + composite_col)
        
        # Use regexp_extract to avoid Fabric runtime issues with column-based substring lengths
        mappings_parsed = (
            mappings_raw
            .withColumn("_composite_key", F.upper(F.trim(F.col(composite_col))))
            .withColumn("_has_dash", F.instr(F.col("_composite_key"), "-") > 0)
            .withColumn(
                "map_identifier",
                F.when(
                    F.col("_has_dash"),
                    F.regexp_extract(F.col("_composite_key"), r"^(.*)-([A-Z]{3})$", 1)
                ).otherwise(
                    F.regexp_extract(F.col("_composite_key"), r"^(.*?)([A-Z]{3})$", 1)
                )
            )
            .withColumn(
                "map_currency",
                F.when(
                    F.col("_has_dash"),
                    F.regexp_extract(F.col("_composite_key"), r"^(.*)-([A-Z]{3})$", 2)
                ).otherwise(
                    F.regexp_extract(F.col("_composite_key"), r"^(.*?)([A-Z]{3})$", 2)
                )
            )
        )
        
        # Now detect the security code column
        security_code_col = None
        for candidate in ["security_code", "final_security_code", "unique_security_code", "mapped_security_id", "target_security_id", "sh_code"]:
            if candidate in scols:
                security_code_col = scols[candidate]
                break
        
        if security_code_col is None:
            print("  [WARNING] Could not find security code column in mappings")
            print("  Using derived_security_id as fallback.")
            enriched = enriched.withColumn("security_code", F.col("derived_security_id"))
        else:
            security_mapping = (
                mappings_parsed
                .select(
                    F.upper(F.trim(F.regexp_replace(F.col("map_identifier"), "\\t", ""))).alias("map_identifier"),
                    F.upper(F.trim(F.regexp_replace(F.col("map_currency"), "\\t", ""))).alias("map_currency"),
                    F.trim(F.col(security_code_col)).alias("mapped_security_code")
                )
                .filter(F.col("map_identifier").isNotNull() & (F.trim(F.col("map_identifier")) != ""))
                .filter(F.col("map_currency").isNotNull() & (F.trim(F.col("map_currency")) != ""))
                .filter(F.col("mapped_security_code").isNotNull() & (F.trim(F.col("mapped_security_code")) != ""))
                .dropDuplicates(["map_identifier", "map_currency"])
            )
            
            sec_map_count = security_mapping.count()
            print(f"  Security mappings (parsed composite key): {sec_map_count} pairs")
            
            # Show sample mappings
            print("  Sample mappings:")
            security_mapping.show(5, truncate=False)
    else:
        # Detect separate identifier and currency columns
        identifier_col = None
        for candidate in ["identifier", "security_id", "sedol", "isin", "asset_id", "source_security_id"]:
            if candidate in scols:
                identifier_col = scols[candidate]
                break

        currency_col = None
        for candidate in ["currency", "ccycode", "ccy", "local_currency"]:
            if candidate in scols:
                currency_col = scols[candidate]
                break

        security_code_col = None
        for candidate in ["security_code", "final_security_code", "unique_security_code", "mapped_security_id", "target_security_id", "sh_code"]:
            if candidate in scols:
                security_code_col = scols[candidate]
                break

        if identifier_col is None or currency_col is None or security_code_col is None:
            print("  [WARNING] Mappings table found but security mapping columns incomplete.")
            print("  Expected identifier like: " + str(['identifier', 'security_id']))
            print("  Expected currency like: " + str(['currency', 'ccy']))
            print("  Expected code like: " + str(['security_code', 'sh_code']))
            print("  Using derived_security_id as fallback.")
            enriched = enriched.withColumn("security_code", F.col("derived_security_id"))
        else:
            # Build composite key security mapping from separate columns
            security_mapping = (
                mappings_raw
                .select(
                    F.upper(F.trim(F.regexp_replace(F.col(identifier_col), "\\t", ""))).alias("map_identifier"),
                    F.upper(F.trim(F.regexp_replace(F.col(currency_col), "\\t", ""))).alias("map_currency"),
                    F.trim(F.col(security_code_col)).alias("mapped_security_code")
                )
                .filter(F.col("map_identifier").isNotNull() & (F.trim(F.col("map_identifier")) != ""))
                .filter(F.col("map_currency").isNotNull() & (F.trim(F.col("map_currency")) != ""))
                .filter(F.col("mapped_security_code").isNotNull() & (F.trim(F.col("mapped_security_code")) != ""))
                .dropDuplicates(["map_identifier", "map_currency"])
            )

            sec_map_count = security_mapping.count()
            print(f"  Security mappings (separate columns): {sec_map_count} pairs")
            
            # Show sample mappings
            print("  Sample mappings:")
            security_mapping.show(5, truncate=False)
    
    # Common JOIN logic for both composite and separate column cases
    if 'security_mapping' in dir():
        print("")
        print("  Joining enriched data with security mappings...")
        total_enriched = enriched.count()
        print("  Enriched records: " + str(total_enriched))
        
        # Show sample derived_security_id values for debugging
        print("  Sample derived_security_id values:")
        enriched.filter(~F.col("_is_cash_line")).select("derived_security_id", "position_currency", "isin", "sedol").show(5, truncate=False)

        # Build normalized join keys to handle tabs/whitespace in currency and identifiers
        enriched = enriched.withColumn(
            "_join_identifier",
            F.upper(F.trim(F.regexp_replace(F.col("derived_security_id"), "\\t", "")))
        ).withColumn(
            "_join_currency",
            F.upper(F.trim(F.regexp_replace(F.col("position_currency"), "\\t", "")))
        )

        security_mapping = security_mapping.withColumn(
            "_map_identifier_norm",
            F.upper(F.trim(F.regexp_replace(F.col("map_identifier"), "\\t", "")))
        ).withColumn(
            "_map_currency_norm",
            F.upper(F.trim(F.regexp_replace(F.col("map_currency"), "\\t", "")))
        )
        
        enriched = enriched.join(
            security_mapping,
            (
                (enriched["_join_identifier"] == security_mapping["_map_identifier_norm"]) &
                (enriched["_join_currency"] == security_mapping["_map_currency_norm"])
            ),
            "left"
        )

        enriched = enriched.withColumn(
            "security_code",
            F.coalesce(F.col("mapped_security_code"), F.col("derived_security_id"))
        )

        mapped_sec_count = enriched.filter(F.col("mapped_security_code").isNotNull()).count()
        print(f"  Security codes mapped: {mapped_sec_count}")
        fallback_count = total_enriched - mapped_sec_count
        print("  Security codes using fallback: " + str(fallback_count))

        enriched = enriched.drop(
            "mapped_security_code", "map_identifier", "map_currency",
            "_join_identifier", "_join_currency", "_map_identifier_norm", "_map_currency_norm"
        )
else:
    enriched = enriched.withColumn("security_code", F.col("derived_security_id"))
    print("  Mappings not available; using derived_security_id as security_code")

# Add placeholder columns for downstream compatibility (TRIM currency to remove tabs/spaces)
enriched = enriched.withColumn("local_currency",F.regexp_replace(F.col("position_currency"), "\\t", ""))
enriched = enriched.withColumn("identifier_chosen", F.col("derived_security_id"))
enriched = enriched.withColumn("asset_name", F.col("instrument_name"))
enriched = enriched.withColumn("id_type", F.lit("sedol_isin"))

print("\n[STEP 5] Security mapping complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 33, Finished, Available, Finished, False)


[STEP 5] Resolving security identifiers and applying security mapping...
  Derived security IDs: 16320 / 16321
  Cash lines (synthetic codes): 805
  Detected mapping columns: ['ubs ref', 'sl polno', 'unnamed: 2', 'isin+ccy', 'final_security_code', 'unique_security_code', 'isin', 'currency', 'sedol', 'security_name']
  Found composite key column: ISIN+CCY
  Parsing composite key from column: ISIN+CCY
  Security mappings (parsed composite key): 9669 pairs
  Sample mappings:
+--------------+------------+--------------------+
|map_identifier|map_currency|mapped_security_code|
+--------------+------------+--------------------+
|AN8068571086  |USD         |2779201             |
|AT0000673165  |EUR         |TPY0002             |
|AT0000674908  |EUR         |BNGX3Q0             |
|AT0000677927  |EUR         |TPY0003             |
|AT0000722640  |EUR         |TPY0004             |
+--------------+------------+--------------------+
only showing top 5 rows


  Joining enriched data with security

In [31]:
# Add this as a new cell AFTER Step 5, BEFORE Step 6
print("Step 5 local_currency diagnostic:")
enriched.select("local_currency").distinct().show(10, truncate=False)
print("\nSample with source columns:")
enriched.select("position_currency", "local_currency", "client_id").show(5, truncate=False)

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 34, Finished, Available, Finished, False)

Step 5 local_currency diagnostic:
+--------------+
|local_currency|
+--------------+
|DKK           |
|GBP           |
|CHF           |
|CAD           |
|EUR           |
|TWD           |
|ZAR           |
|JPY           |
|USD           |
|HKD           |
+--------------+
only showing top 10 rows


Sample with source columns:
+-----------------+--------------+-----------+
|position_currency|local_currency|client_id  |
+-----------------+--------------+-----------+
|\tUSD            |USD           |ID-70471336|
|\tGBP            |GBP           |ID-A1DBC22D|
|\tEUR            |EUR           |ID-6CAF0219|
|\tGBP            |GBP           |ID-6CAF0219|
|\tGBP            |GBP           |ID-6CAF0219|
+-----------------+--------------+-----------+
only showing top 5 rows



## Step 6: Lookup Validation

Binary validation equivalent to Excel Step 5 (`#N/A` checks). This identifies missing policy/security/asset mappings before value conversion and downstream checks.

In [32]:
# ========== Step 6: Lookup Validation (Excel Step 5 Equivalent) ==========
print("\n[STEP 6] Checking lookup completeness (no #N/A equivalent)...")
print("=" * 70)

missing_policy_count = enriched.filter(F.col("policyholder_number").isNull() | (F.trim(F.col("policyholder_number")) == "")).count()
missing_identifier_count = enriched.filter(F.col("security_code").isNull() | (F.trim(F.col("security_code")) == "")).count()
missing_asset_name_count = enriched.filter(
    (~F.col("_is_cash_line")) & (F.col("asset_name").isNull() | (F.trim(F.col("asset_name")) == ""))
).count()

print(f"  Missing policy mappings: {missing_policy_count}")
print(f"  Missing security identifiers: {missing_identifier_count}")
print(f"  Missing asset names (non-cash): {missing_asset_name_count}")

if (missing_policy_count + missing_identifier_count + missing_asset_name_count) > 0:
    print("\n  [HITL REQUIRED] Lookup gaps detected. Review exception samples below.")

    print("\n  Sample missing policies:")
    enriched.filter(F.col("policyholder_number").isNull() | (F.trim(F.col("policyholder_number")) == "")) \
        .select("client_id", "source_file", "source_row_id") \
        .show(10, truncate=False)

    print("\n  Sample missing security identifiers:")
    enriched.filter(F.col("security_code").isNull() | (F.trim(F.col("security_code")) == "")) \
        .select("client_id", "instrument_name", "isin", "sedol", "source_row_id") \
        .show(10, truncate=False)

    print("\n  Sample missing asset names:")
    enriched.filter((~F.col("_is_cash_line")) & (F.col("asset_name").isNull() | (F.trim(F.col("asset_name")) == ""))) \
        .select("client_id", "security_code", "instrument_name", "source_row_id") \
        .show(10, truncate=False)
else:
    print("\n  [PASS] All key lookups populated.")

print("\n[STEP 6] Lookup validation complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 35, Finished, Available, Finished, False)


[STEP 6] Checking lookup completeness (no #N/A equivalent)...
  Missing policy mappings: 1
  Missing security identifiers: 1
  Missing asset names (non-cash): 0

  [HITL REQUIRED] Lookup gaps detected. Review exception samples below.

  Sample missing policies:
+---------+-----------------------+------------------------------+
|client_id|source_file            |source_row_id                 |
+---------+-----------------------+------------------------------+
|         |Notification - Cash.csv|Notification - Cash.csv:CASH:1|
+---------+-----------------------+------------------------------+


  Sample missing security identifiers:
+-----------+------------------------+----+-----+--------------------------+
|client_id  |instrument_name         |isin|sedol|source_row_id             |
+-----------+------------------------+----+-----+--------------------------+
|ID-E338B0A7|GS DEF AUTOCALL WO 8.40%|    |     |Notification.csv:POS:12511|
+-----------+------------------------+----+-----+----

## Step 7: Convert Values and Apply FX

Parse numeric values, apply FX rates, calculate GBP equivalents.

In [33]:
# ========== Value Conversion and FX Application ==========
print("\n[STEP 7] Converting values and applying FX rates...")
print("=" * 70)

# Helper: convert European decimal format "1.234,56" to "1234.56"
def parse_euro_decimal(col_name):
    return F.regexp_replace(
        F.regexp_replace(F.trim(F.col(col_name)), r"\.", ""),
        ",", "."
    ).cast("double")

# Parse position values
enriched = enriched.withColumn("bid_value_local", parse_euro_decimal("balance_local"))
enriched = enriched.withColumn("accrued_interest_local", parse_euro_decimal("accrued_interest_local"))

# Parse cash values
enriched = enriched.withColumn("cash_value_local", parse_euro_decimal("cash_balance_local"))

# Apply FX rates
if fx_rates is not None:
    enriched = enriched.join(
       fx_rates,
       F.regexp_replace(enriched["local_currency"], "\\t", "") == fx_rates["currency"],  # Strip tabs in join
      "left"
)
    
    # For GBP, rate = 1.0
    enriched = enriched.withColumn(
        "fx_rate",
        F.when(F.col("local_currency") == "GBP", F.lit(1.0))
         .otherwise(F.col("fx_rate"))
    )
else:
    # No FX rates - assume everything is GBP
    enriched = enriched.withColumn("fx_rate", F.lit(1.0))

# Calculate GBP values
enriched = enriched.withColumn(
    "bid_value_gbp",
    F.when(F.col("bid_value_local").isNull(), F.lit(None).cast("double"))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("bid_value_local") * F.col("fx_rate"))
)

enriched = enriched.withColumn(
    "cash_value_gbp",
    F.when(F.col("cash_value_local").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("cash_value_local") * F.col("fx_rate"))
)

enriched = enriched.withColumn(
    "accrued_interest_gbp",
    F.when(F.col("accrued_interest_local").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("accrued_interest_local") * F.col("fx_rate"))
)

# Holdings and price (not available in Brown Shipley source)
enriched = enriched.withColumn("holding", F.lit(None).cast("double"))
enriched = enriched.withColumn("local_bid_price", F.lit(None).cast("double"))

print("  Value conversion complete")
print("  FX rates applied")
print("  GBP equivalents calculated")

print("\n[STEP 7] Value conversion and FX application complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 36, Finished, Available, Finished, False)


[STEP 7] Converting values and applying FX rates...
  Value conversion complete
  FX rates applied
  GBP equivalents calculated

[STEP 7] Value conversion and FX application complete


In [34]:
# ========== Step 7A: Workbook Parity Override for L/N/O ==========
print("\n[STEP 7A] Recomputing workbook-equivalent L/N/O fields...")
print("=" * 70)

# Parse mixed numeric formats safely.
def parse_mixed_decimal(col_name):
    raw = F.regexp_replace(F.coalesce(F.col(col_name).cast("string"), F.lit("")), "\u00A0", "")
    raw = F.regexp_replace(raw, r"\t", "")
    raw = F.regexp_replace(raw, r"\s+", "")

    both_sep = (F.instr(raw, ".") > 0) & (F.instr(raw, ",") > 0)
    comma_only = (F.instr(raw, ",") > 0) & (F.instr(raw, ".") == 0)
    comma_thousands = raw.rlike(r"^-?[0-9]{1,3}(,[0-9]{3})+$")

    normalized = (
        F.when(raw == "", F.lit(None).cast("string"))
         .when(both_sep, F.regexp_replace(F.regexp_replace(raw, r"\.", ""), ",", "."))
         .when(comma_only & comma_thousands, F.regexp_replace(raw, ",", ""))
         .when(comma_only, F.regexp_replace(raw, ",", "."))
         .otherwise(raw)
    )
    return normalized.cast("double")

# Ensure required source columns exist.
if "position_value_local_raw" not in enriched.columns:
    enriched = enriched.withColumn("position_value_local_raw", F.col("balance_local").cast("string"))
if "position_local_bid_price_raw" not in enriched.columns:
    enriched = enriched.withColumn("position_local_bid_price_raw", F.lit(None).cast("string"))

# N: Holding
enriched = enriched.withColumn("holding", parse_mixed_decimal("balance_local"))

# L local input: prefer workbook source value column, fallback to holding.
enriched = enriched.withColumn("position_value_local_num", parse_mixed_decimal("position_value_local_raw"))
enriched = enriched.withColumn(
    "bid_value_local",
    F.when(F.col("source_record_type") == "POSITION", F.coalesce(F.col("position_value_local_num"), F.col("holding")))
     .otherwise(F.lit(None).cast("double"))
)

enriched = enriched.withColumn("cash_value_local", parse_mixed_decimal("cash_balance_local"))
enriched = enriched.withColumn("accrued_interest_local_num", parse_mixed_decimal("accrued_interest_local"))

# L: Bid_Value_in_GBP
enriched = enriched.withColumn(
    "bid_value_gbp",
    F.when(F.col("source_record_type") == "POSITION",
           F.when(F.col("bid_value_local").isNull() | F.col("fx_rate").isNull(), F.lit(None).cast("double"))
            .otherwise(F.col("bid_value_local") * F.col("fx_rate")))
     .otherwise(F.lit(None).cast("double"))
)

# K: Cash_Value_in_GBP
enriched = enriched.withColumn(
    "cash_value_gbp",
    F.when(F.col("source_record_type") == "CASH",
           F.when(F.col("cash_value_local").isNull() | F.col("fx_rate").isNull(), F.lit(None).cast("double"))
            .otherwise(F.col("cash_value_local") * F.col("fx_rate")))
     .otherwise(F.lit(0.0))
)

enriched = enriched.withColumn(
    "accrued_interest_gbp",
    F.when(F.col("accrued_interest_local_num").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("accrued_interest_local_num") * F.col("fx_rate"))
)

# O: Loc_Bid_Price. Cash/TPY_CASH rows blank; otherwise (L/N)/FX with safeguards.
enriched = enriched.withColumn("local_bid_price_source", parse_mixed_decimal("position_local_bid_price_raw"))
enriched = enriched.withColumn(
    "local_bid_price",
    F.when(
        F.col("security_code").contains("TPY_CASH") |
        F.col("security_code").startswith("CASH_") |
        (F.col("source_record_type") == "CASH"),
        F.lit(None).cast("double")
    )
    .when(F.col("local_bid_price_source").isNotNull(), F.col("local_bid_price_source"))
    .when(
        F.col("holding").isNull() |
        (F.col("holding") == 0) |
        F.col("fx_rate").isNull() |
        (F.col("fx_rate") == 0) |
        F.col("bid_value_gbp").isNull(),
        F.lit(None).cast("double")
    )
    .otherwise((F.col("bid_value_gbp") / F.col("holding")) / F.col("fx_rate"))
)

print("  Recomputed: holding, bid_value_local, bid_value_gbp, cash_value_gbp, local_bid_price")
print("[STEP 7A] Workbook parity override complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 37, Finished, Available, Finished, False)


[STEP 7A] Recomputing workbook-equivalent L/N/O fields...
  Recomputed: holding, bid_value_local, bid_value_gbp, cash_value_gbp, local_bid_price
[STEP 7A] Workbook parity override complete


In [35]:
# After Step 7 (Value Conversion), run this:
print("Policy numbers and bid values:")
enriched.select(
    "policyholder_number", 
    "bid_value_gbp"
).filter(
    F.col("policyholder_number").isNotNull() & 
    (F.trim(F.col("policyholder_number")) != "")
).groupBy("policyholder_number").agg(
    F.count(F.lit(1)).alias("row_count"),
    F.sum(F.col("bid_value_gbp")).alias("total_bid_value_gbp")
).orderBy(F.desc("total_bid_value_gbp")).show(30)

print("\nCount of non-blank, non-zero bid values:")
enriched.filter(
    (F.col("policyholder_number").isNotNull()) &
    (F.trim(F.col("policyholder_number")) != "") &
    (F.col("bid_value_gbp").isNotNull()) &
    (F.col("bid_value_gbp") > 0)
).count()

print("\nIH Report matches:")
enriched.filter(
    F.col("ih_lookup_count") > 0
).count()

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 38, Finished, Available, Finished, False)

Policy numbers and bid values:
+-------------------+---------+--------------------+
|policyholder_number|row_count| total_bid_value_gbp|
+-------------------+---------+--------------------+
|        ID-EE145E7D|      174|8.494247417246477E10|
|        ID-8CA93073|      181|7.250464272653969E10|
|        ID-B17AECB8|      181| 7.754476507088098E9|
|        ID-4BFE4413|     2991| 2.809743081363779E7|
|        ID-1182608C|      196|1.2010595425706258E7|
|        ID-CC2DEC95|      120|   9856400.498151572|
|        ID-0F593B45|      127|   9843839.498186896|
|        ID-B979120D|      152|   9730168.498877909|
|        ID-2E4AE09B|       95|   8897983.258534215|
|        ID-5BB18400|       98|   7987382.011650198|
|        ID-82BA3F24|      177|  7479492.7487915745|
|        ID-BA6E5D99|       97|   6346237.563752489|
|        ID-9177E07A|       97|    6300589.13846419|
|        ID-69626DDA|       96|     4869092.0060381|
|        ID-74BAF56B|      195|   4639584.155320762|
|        ID-D0A

9070

In [36]:
# After Step 7, check the components:
enriched.select(
    "client_id",
    "balance_local",
    "bid_value_local",
    "local_currency",
    "fx_rate",
    "bid_value_gbp"
).filter(F.col("bid_value_local").isNotNull()).show(5, truncate=False)

# Correct null count syntax:
print("Null counts:")
enriched.select(
    F.sum(F.when(F.col("balance_local").isNull(), 1).otherwise(0)).alias("balance_local_nulls"),
    F.sum(F.when(F.col("bid_value_local").isNull(), 1).otherwise(0)).alias("bid_value_local_nulls"),
    F.sum(F.when(F.col("fx_rate").isNull(), 1).otherwise(0)).alias("fx_rate_nulls"),
    F.sum(F.when(F.col("bid_value_gbp").isNull(), 1).otherwise(0)).alias("bid_value_gbp_nulls"),
    F.count(F.lit(1)).alias("total_rows")
).show()

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 39, Finished, Available, Finished, False)

+-----------+-------------+---------------+--------------+----------------+-----------------+
|client_id  |balance_local|bid_value_local|local_currency|fx_rate         |bid_value_gbp    |
+-----------+-------------+---------------+--------------+----------------+-----------------+
|ID-2760771C|0,00         |0.0            |GBP           |1.0             |0.0              |
|ID-510048B1|0,00         |0.0            |GBP           |1.0             |0.0              |
|ID-35CA8CD0|0,00         |0.0            |GBP           |1.0             |0.0              |
|ID-923CAF90|0,00         |0.0            |GBP           |1.0             |0.0              |
|ID-190F5ED3|432,00       |140508.0       |DKK           |0.11685655857435|16419.28133216477|
+-----------+-------------+---------------+--------------+----------------+-----------------+
only showing top 5 rows

Null counts:
+-------------------+---------------------+-------------+-------------------+----------+
|balance_local_nulls|bid_va

## Step 8: Include/Remove Flagging

Apply Brown Shipley workbook-equivalent decision table (from edited sheet formula):
- Rule 1: bid_value_in_GBP = 0 or blank → `Remove`
- Rule 2: policy blank or `REMOVE*` → `Remove`
- Rule 3: policy exists in IH Report → `Include`
- Rule 4: default → `Remove`

For compatibility with downstream notebooks, both `include` (workbook label) and `include_flag` (pipeline field) are populated.

In [37]:
# ========== Include/Remove Flagging ==========
print("\n[STEP 8] Applying Include/Remove logic...")
print("=" * 70)

# Workbook-equivalent rule fields
enriched = enriched.withColumn(
    "policyholder_number_clean",
    F.upper(F.trim(F.coalesce(F.col("policyholder_number"), F.lit(""))))
)

enriched = enriched.withColumn("ih_policy_exists", F.col("ih_lookup_count") > 0)

# Excel decision table precedence (from edited sheet formula):
# 1) bid_value_in_GBP = 0 or blank -> Remove
# 2) Blank policy -> Remove
# 3) Literal REMOVE* marker -> Remove
# 4) Policy exists in IH Report -> Include
# 5) Default -> Remove
enriched = enriched.withColumn(
    "include",
    F.when(
        F.col("bid_value_gbp").isNull() | (F.col("bid_value_gbp") == 0),
        F.lit("Remove")
    )
    .when(
        F.col("policyholder_number_clean") == "",
        F.lit("Remove")
    )
    .when(
        F.col("policyholder_number_clean") == "REMOVE*",
        F.lit("Remove")
    )
    .when(
        F.col("ih_policy_exists"),
        F.lit("Include")
    )
    .otherwise(F.lit("Remove"))
)

# Keep pipeline-compatible include flag
enriched = enriched.withColumn(
    "include_flag",
    F.when(F.col("include") == "Include", F.lit("Include")).otherwise(F.lit("Remove"))
)

# Reason codes for traceability
enriched = enriched.withColumn(
    "exclusion_reason_code",
    F.when(
        F.col("bid_value_gbp").isNull() | (F.col("bid_value_gbp") == 0),
        F.lit("REMOVE_ZERO_VALUE")
    )
    .when(
        F.col("policyholder_number_clean") == "",
        F.lit("REMOVE_BLANK_POLICY")
    )
    .when(
        F.col("policyholder_number_clean") == "REMOVE*",
        F.lit("REMOVE_LITERAL_MARKER")
    )
    .when(
        ~F.col("ih_policy_exists"),
        F.lit("REMOVE_NOT_IN_IH")
    )
    .otherwise(F.lit(None).cast("string"))
)

# Distributions
include_dist = enriched.groupBy("include").count().orderBy("include")
print("  Include/Remove distribution (workbook label):")
for row in include_dist.collect():
    print(f"    {row['include']}: {row['count']} rows")

pipeline_include_dist = enriched.groupBy("include_flag").count().orderBy("include_flag")
print("\n  include_flag distribution (pipeline):")
for row in pipeline_include_dist.collect():
    print(f"    {row['include_flag']}: {row['count']} rows")

exclusion_reasons = enriched.filter(F.col("include_flag") == "Remove") \
    .groupBy("exclusion_reason_code").count().orderBy(F.desc("count"))
if exclusion_reasons.count() > 0:
    print("\n  Exclusion reasons:")
    for row in exclusion_reasons.collect():
        print(f"    {row['exclusion_reason_code']}: {row['count']} rows")

print("\n[STEP 8] Include/Remove flagging complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 40, Finished, Available, Finished, False)


[STEP 8] Applying Include/Remove logic...
  Include/Remove distribution (workbook label):
    Include: 2400 rows
    Remove: 13921 rows

  include_flag distribution (pipeline):
    Include: 2400 rows
    Remove: 13921 rows

  Exclusion reasons:
    REMOVE_ZERO_VALUE: 12920 rows
    REMOVE_NOT_IN_IH: 1001 rows

[STEP 8] Include/Remove flagging complete


In [38]:
# ========== Step 8B: Workbook Include/Remove Value Basis Override ==========
print("\n[STEP 8B] Aligning Include/Remove value basis to workbook...")
print("=" * 70)

# Workbook parity: cash rows evaluate with K (cash_value_gbp), non-cash rows with L (bid_value_gbp).
enriched = enriched.withColumn(
    "value_for_include_gbp",
    F.when(F.col("source_record_type") == "CASH", F.col("cash_value_gbp"))
     .otherwise(F.col("bid_value_gbp"))
)

enriched = enriched.withColumn(
    "include",
    F.when(
        F.col("value_for_include_gbp").isNull() | (F.col("value_for_include_gbp") == 0),
        F.lit("Remove")
    )
    .when(
        F.col("policyholder_number_clean") == "",
        F.lit("Remove")
    )
    .when(
        F.col("policyholder_number_clean") == "REMOVE*",
        F.lit("Remove")
    )
    .when(
        F.col("ih_policy_exists"),
        F.lit("Include")
    )
    .otherwise(F.lit("Remove"))
)

enriched = enriched.withColumn(
    "include_flag",
    F.when(F.col("include") == "Include", F.lit("Include")).otherwise(F.lit("Remove"))
)

enriched = enriched.withColumn(
    "exclusion_reason_code",
    F.when(
        F.col("value_for_include_gbp").isNull() | (F.col("value_for_include_gbp") == 0),
        F.lit("REMOVE_ZERO_VALUE")
    )
    .when(
        F.col("policyholder_number_clean") == "",
        F.lit("REMOVE_BLANK_POLICY")
    )
    .when(
        F.col("policyholder_number_clean") == "REMOVE*",
        F.lit("REMOVE_LITERAL_MARKER")
    )
    .when(
        ~F.col("ih_policy_exists"),
        F.lit("REMOVE_NOT_IN_IH")
    )
    .otherwise(F.lit(None).cast("string"))
)

pipeline_include_dist = enriched.groupBy("include_flag").count().orderBy("include_flag")
print("  include_flag distribution after workbook basis override:")
for row in pipeline_include_dist.collect():
    print(f"    {row['include_flag']}: {row['count']} rows")

print("[STEP 8B] Include/Remove basis override complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 41, Finished, Available, Finished, False)


[STEP 8B] Aligning Include/Remove value basis to workbook...
  include_flag distribution after workbook basis override:
    Include: 2559 rows
    Remove: 13762 rows
[STEP 8B] Include/Remove basis override complete


## Step 9: Add Check Columns and Decision Trace

Add validation check flags and decision traceability.

In [39]:
# ========== Check Columns and Decision Trace ==========
print("\n[STEP 9] Adding check columns and decision trace...")
print("=" * 70)

# DFM Step 7 check uses IH Report Movt% tolerance (98-102).
# Movt may arrive as '101%', '101.2', or null.
enriched = enriched.withColumn(
    "movt_percent",
    F.regexp_replace(F.trim(F.coalesce(F.col("movt_raw"), F.lit(""))), "%", "").cast("double")
)

# Movt-based holdings check (replaces unavailable holding x price check for this DFM).
enriched = enriched.withColumn(
    "holdings_check_flag",
    F.when(F.col("movt_percent").isNull(), F.lit("not_evaluable"))
     .when((F.col("movt_percent") >= 98.0) & (F.col("movt_percent") <= 102.0), F.lit("pass"))
     .otherwise(F.lit("fail"))
)

# Acquisition value check (stub - not available in source)
enriched = enriched.withColumn("acq_value_check_flag", F.lit("not_evaluable"))

# Decision trace JSON
enriched = enriched.withColumn(
    "decision_trace_json",
    F.to_json(
        F.struct(
            F.col("client_id").alias("policy_original"),
            F.col("policyholder_number").alias("policy_mapped"),
            F.col("policy_mapping_applied"),
            F.col("ih_policy_exists"),
            F.col("ih_lookup_count"),
            F.col("bid_value_gbp"),
            F.col("include"),
            F.col("include_flag"),
            F.col("identifier_chosen"),
            F.col("sedol").alias("source_sedol"),
            F.col("isin").alias("source_isin"),
            F.col("security_code"),
            F.col("exclusion_reason_code"),
            F.col("movt_percent"),
            F.col("holdings_check_flag")
        )
    )
)

# Data quality flags
enriched = enriched.withColumn(
    "data_quality_flags",
    F.array(
        F.when(F.col("fx_rate").isNull(), F.lit("FX_NOT_AVAILABLE")),
        F.when(F.col("policy_mapping_applied") == False, F.lit("POLICY_NOT_MAPPED")),
        F.when(F.col("movt_percent").isNull(), F.lit("MOVT_NOT_AVAILABLE")),
        F.when(F.col("holdings_check_flag") == "fail", F.lit("MOVT_OUTSIDE_TOLERANCE"))
    ).cast("array<string>")
)

# Remove nulls from array
enriched = enriched.withColumn(
    "data_quality_flags",
    F.expr("filter(data_quality_flags, x -> x is not null)")
)

check_dist = enriched.groupBy("holdings_check_flag").count().orderBy("holdings_check_flag")
print("  Movt tolerance check distribution:")
for row in check_dist.collect():
    print(f"    {row['holdings_check_flag']}: {row['count']} rows")

failed_movt = enriched.filter(F.col("holdings_check_flag") == "fail")
if failed_movt.count() > 0:
    print("\n  Sample Movt failures (outside 98-102):")
    failed_movt.select("client_id", "movt_raw", "movt_percent", "include_flag").show(10, truncate=False)

print("  Check columns added")
print("  Decision trace JSON generated")
print("  Data quality flags populated")

print("\n[STEP 9] Check columns and decision trace complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 42, Finished, Available, Finished, False)


[STEP 9] Adding check columns and decision trace...
  Movt tolerance check distribution:
    not_evaluable: 16321 rows
  Check columns added
  Decision trace JSON generated
  Data quality flags populated

[STEP 9] Check columns and decision trace complete


## Step 8A: Build HITL Review Queues

Creates review datasets for human-in-the-loop decisions:
- Step 6 missing item: missing security mapping
- Step 6 missing item: missing policy mapping
- Step 8 missing item: `Movt` anomalies requiring review

In [40]:
# ========== Step 8A: Build HITL Review Queues ==========
print("\n[STEP 8A] Building HITL review datasets...")
print("=" * 70)

# Step 6a queue: missing security mapping context for manual enrichment.
queue_missing_security = enriched.filter(
    (~F.col("_is_cash_line")) &
    (
        F.col("security_code").isNull() |
        (F.trim(F.col("security_code")) == "") |
        F.col("asset_name").isNull() |
        (F.trim(F.col("asset_name")) == "")
    )
).select(
    "period", "run_id", "client_id", "instrument_name", "isin", "sedol", "security_code", "asset_name", "source_file", "source_row_id"
)

# Step 6b queue: missing policy mapping cases for manual policy resolution.
queue_missing_policy = enriched.filter(
    F.col("policyholder_number").isNull() | (F.trim(F.col("policyholder_number")) == "")
).select(
    "period", "run_id", "client_id", "instrument_name", "security_code", "movt_raw", "source_file", "source_row_id"
)

# Step 8 queue: Movt anomalies that need human review/escalation.
queue_movt_anomalies = enriched.filter(
    F.col("holdings_check_flag") == "fail"
).select(
    "period", "run_id", "client_id", "policyholder_number", "movt_raw", "movt_percent", "include_flag", "source_file", "source_row_id"
)

missing_security_count = queue_missing_security.count()
missing_policy_count = queue_missing_policy.count()
movt_anomaly_count = queue_movt_anomalies.count()

print(f"  Step 6a queue (missing security mapping): {missing_security_count}")
print(f"  Step 6b queue (missing policy mapping): {missing_policy_count}")
print(f"  Step 8 queue (Movt anomaly / policy review): {movt_anomaly_count}")

if missing_security_count > 0:
    print("\n  Sample Step 6a rows:")
    queue_missing_security.show(10, truncate=False)

if missing_policy_count > 0:
    print("\n  Sample Step 6b rows:")
    queue_missing_policy.show(10, truncate=False)

if movt_anomaly_count > 0:
    print("\n  Sample Step 8 rows:")
    queue_movt_anomalies.show(10, truncate=False)

# Optional persistence for operational review.
# Uncomment if you want durable queues:
# queue_missing_security.write.mode("overwrite").saveAsTable("stage2_review_queue_missing_security")
# queue_missing_policy.write.mode("overwrite").saveAsTable("stage2_review_queue_missing_policy")
# queue_movt_anomalies.write.mode("overwrite").saveAsTable("stage2_review_queue_movt_anomalies")

print("\n[STEP 8A] HITL review queues ready")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 43, Finished, Available, Finished, False)


[STEP 8A] Building HITL review datasets...
  Step 6a queue (missing security mapping): 1
  Step 6b queue (missing policy mapping): 1
  Step 8 queue (Movt anomaly / policy review): 0

  Sample Step 6a rows:
+-------+---------------+-----------+------------------------+----+-----+-------------+------------------------+----------------+--------------------------+
|period |run_id         |client_id  |instrument_name         |isin|sedol|security_code|asset_name              |source_file     |source_row_id             |
+-------+---------------+-----------+------------------------+----+-----+-------------+------------------------+----------------+--------------------------+
|2026-03|manual_test_run|ID-E338B0A7|GS DEF AUTOCALL WO 8.40%|    |     |NULL         |GS DEF AUTOCALL WO 8.40%|Notification.csv|Notification.csv:POS:12511|
+-------+---------------+-----------+------------------------+----+-----+-------------+------------------------+----------------+--------------------------+


  Samp

## Step 9: Project to Stage 2 Contract

Project to `individual_dfm_consolidated` schema.

In [41]:
# ========== Project to Stage 2 Contract ==========
print("\n[STEP 9] Projecting to individual_dfm_consolidated schema...")
print("=" * 70)

# Generate row hash for deduplication
# Include final calculated values to detect true duplicates vs. data corrections
enriched = enriched.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("period"),
            F.col("dfm_id"),
            F.col("policyholder_number"),
            F.coalesce(F.col("security_code"), F.lit("")),
            F.coalesce(F.col("isin"), F.lit("")),
            F.coalesce(F.col("source_row_id"), F.lit("")),
            F.coalesce(F.col("bid_value_gbp").cast("string"), F.lit("")),
            F.coalesce(F.col("cash_value_gbp").cast("string"), F.lit(""))
        ),
        256
    )
)

# Parse value_date to date
enriched = enriched.withColumn(
    "report_date",
    F.coalesce(
        F.to_date(F.col("value_date"), "dd/MM/yyyy"),
        F.to_date(F.col("value_date"), "yyyy-MM-dd"),
        F.to_date(F.col("value_date"))
    )
)

# Select Stage 2 contract columns
stage2_output = enriched.select(
    # Provenance
    F.col("period"),
    F.col("run_id"),
    F.col("dfm_id"),
    F.lit(profile_id).alias("profile_id"),
    F.col("source_file"),
    F.col("source_row_id"),
    F.col("row_hash"),
    
    # Policy identifiers
    F.col("policyholder_number"),
    
    # Security identifiers
    F.col("security_code"),
    F.upper(F.trim(F.col("isin"))).alias("isin"),
    F.upper(F.trim(F.col("sedol"))).alias("sedol"),
    F.lit(None).cast("string").alias("other_security_id"),
    F.col("id_type"),
    F.col("asset_name"),
    
    # Position values
    F.col("holding").cast("decimal(28,10)"),
    F.col("local_bid_price").cast("decimal(28,10)"),
    F.col("local_currency"),
    F.col("fx_rate").cast("decimal(28,10)"),
    F.col("cash_value_gbp").cast("decimal(28,10)"),
    F.col("bid_value_gbp").cast("decimal(28,10)"),
    F.col("accrued_interest_gbp").cast("decimal(28,10)"),
    
    # Include/Remove
    F.col("include_flag"),
    F.col("exclusion_reason_code"),
    
    # Decision trace
    F.col("identifier_chosen"),
    F.col("decision_trace_json"),
    F.col("data_quality_flags"),
    
    # Metadata
    F.col("report_date"),
    F.current_timestamp().alias("transformed_at")
)

# Remove duplicates based on row_hash
stage2_output = stage2_output.dropDuplicates(["row_hash"])

output_count = stage2_output.count()
print(f"  Stage 2 output rows: {output_count}")
print(f"  Input rows (Stage 1): {combined_count}")

if output_count != combined_count:
    print(f"  [WARNING] Row count changed. Expected {combined_count}, got {output_count}")

print("\n[STEP 9] Schema projection complete")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 44, Finished, Available, Finished, False)


[STEP 9] Projecting to individual_dfm_consolidated schema...
  Stage 2 output rows: 16321
  Input rows (Stage 1): 16070
  [WARNING] Row count changed. Expected 16070, got 16321

[STEP 9] Schema projection complete


## Step 10: Write to individual_dfm_consolidated

Persist Stage 2 output table.

In [42]:
# ========== Write Stage 2 Output ==========
print("\n[STEP 10] Writing to individual_dfm_consolidated table...")
print("=" * 70)

try:
    stage2_output.write.mode("append").saveAsTable("individual_dfm_consolidated")
    print(f"  ✓ Successfully wrote {output_count} rows to individual_dfm_consolidated")
except Exception as e:
    print(f"  [ERROR] Failed to write to individual_dfm_consolidated: {e}")
    mssparkutils.notebook.exit("FAILED")

print("\n[STEP 10] Stage 2 output written successfully")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 45, Finished, Available, Finished, False)


[STEP 10] Writing to individual_dfm_consolidated table...
  ✓ Successfully wrote 16321 rows to individual_dfm_consolidated

[STEP 10] Stage 2 output written successfully


## Summary and Validation

Final statistics and validation checks.

In [43]:
# ========== Summary ==========
print("\n" + "=" * 70)
print("STAGE 2 TRANSFORMATION COMPLETE")
print("=" * 70)

print(f"\nPeriod: {period}")
print(f"Run ID: {run_id}")
print(f"DFM: {dfm_name}")

print(f"\nRow Counts:")
print(f"  Stage 1 input (positions): {positions_count}")
if cash is not None:
    print(f"  Stage 1 input (cash): {cash_count}")
print(f"  Combined: {combined_count}")
print(f"  Stage 2 output: {output_count}")

include_summary = stage2_output.groupBy("include_flag").count().collect()
print(f"\nInclude/Remove Summary:")
for row in include_summary:
    print(f"  {row['include_flag']}: {row['count']} rows")

print(f"\nNext Steps:")
print(f"  1. Run nb_stage3_tpir_projection to create tpir_load_equivalent")
print(f"  2. Run nb_validate to generate dq_results and dq_exception_rows")
print(f"  3. Review validation results before publish")

print("\n" + "=" * 70)
mssparkutils.notebook.exit("OK")

StatementMeta(, 084a12c4-8116-4589-9ce0-ed53766f9637, 46, Finished, Available, Finished, False)


STAGE 2 TRANSFORMATION COMPLETE

Period: 2026-03
Run ID: manual_test_run
DFM: Brown Shipley

Row Counts:
  Stage 1 input (positions): 15274
  Stage 1 input (cash): 796
  Combined: 16070
  Stage 2 output: 16321

Include/Remove Summary:
  Remove: 13762 rows
  Include: 2559 rows

Next Steps:
  1. Run nb_stage3_tpir_projection to create tpir_load_equivalent
  2. Run nb_validate to generate dq_results and dq_exception_rows
  3. Review validation results before publish

ExitValue: OK

In [4]:
%%sql
-- ============================================================
-- STEP 1: Run this SQL in Fabric SQL Notebook Cell
-- ============================================================

-- Create the Reconciliation Validation View
CREATE OR REPLACE VIEW vw_holdings_price_validation AS
WITH base AS (
    SELECT
        source_row_id,
        policyholder_number,
        security_code,
        asset_name,
        local_currency AS currency_code,
        include_flag,
        holding,
        local_bid_price,
        bid_value_gbp,
        CASE
            WHEN bid_value_gbp IS NULL OR fx_rate IS NULL OR fx_rate = 0 THEN NULL
            ELSE CAST(bid_value_gbp AS DOUBLE) / CAST(fx_rate AS DOUBLE)
        END AS bid_value_local,
        period,
        run_id,
        id_type,
        isin,
        sedol,
        other_security_id,
        accrued_interest_gbp,
        cash_value_gbp,
        fx_rate,
        exclusion_reason_code,
        identifier_chosen
    FROM individual_dfm_consolidated
    WHERE security_code IS NOT NULL
      AND TRIM(security_code) != ''
)
SELECT
    -- Identification
    source_row_id,
    policyholder_number,
    security_code,
    asset_name,

    -- Lookup context
    currency_code,
    include_flag,

    -- Core metrics
    holding,
    local_bid_price,
    bid_value_local,
    bid_value_gbp,

    -- Reconciliation calculation
    CASE
        WHEN holding IS NULL OR local_bid_price IS NULL OR bid_value_local IS NULL THEN NULL
        WHEN bid_value_local = 0 THEN NULL
        ELSE ROUND((CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0, 2)
    END AS reconciliation_pct,

    -- Outlier flag (outside 98-102% range)
    CASE
        WHEN holding IS NULL OR local_bid_price IS NULL OR bid_value_local IS NULL THEN 'MISSING_DATA'
        WHEN bid_value_local = 0 THEN 'ZERO_VALUE'
        WHEN (CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0 < 98.0 THEN 'BELOW_RANGE'
        WHEN (CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0 > 102.0 THEN 'ABOVE_RANGE'
        ELSE 'IN_RANGE'
    END AS reconciliation_status,

    -- Variance from expected (100%)
    CASE
        WHEN holding IS NULL OR local_bid_price IS NULL OR bid_value_local IS NULL THEN NULL
        WHEN bid_value_local = 0 THEN NULL
        ELSE ROUND(((CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0) - 100.0, 2)
    END AS variance_from_100_pct,

    -- Data completeness flags
    CASE WHEN holding IS NULL THEN 1 ELSE 0 END AS is_holding_null,
    CASE WHEN local_bid_price IS NULL THEN 1 ELSE 0 END AS is_price_null,
    CASE WHEN bid_value_local IS NULL THEN 1 ELSE 0 END AS is_bid_value_null,

    -- Metadata
    period,
    run_id,
    id_type,
    isin,
    sedol,
    other_security_id,
    accrued_interest_gbp,
    cash_value_gbp,
    fx_rate,
    exclusion_reason_code,
    identifier_chosen,
    current_timestamp() AS view_refresh_time
FROM base;

-- Verify view was created
SELECT COUNT(*) as record_count FROM vw_holdings_price_validation;

-- ============================================================
-- STEP 2: Preview the reconciliation metrics (first 1000 rows)
-- ============================================================

SELECT
    policyholder_number,
    security_code,
    asset_name,
    holding,
    local_bid_price,
    bid_value_local,
    reconciliation_pct,
    reconciliation_status,
    variance_from_100_pct,
    currency_code,
    include_flag
FROM vw_holdings_price_validation
ORDER BY ABS(variance_from_100_pct) DESC
LIMIT 1000;

-- ============================================================
-- STEP 3: Check reconciliation summary statistics
-- ============================================================

SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) AS records_in_range,
    SUM(CASE WHEN reconciliation_status != 'IN_RANGE' THEN 1 ELSE 0 END) AS outlier_records,
    ROUND(100.0 * SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_in_range,
    MIN(reconciliation_pct) AS min_reconciliation_pct,
    MAX(reconciliation_pct) AS max_reconciliation_pct,
    ROUND(AVG(reconciliation_pct), 2) AS avg_reconciliation_pct
FROM vw_holdings_price_validation;

-- ============================================================
-- STEP 4: Breakdown by reconciliation status
-- ============================================================

SELECT 
    reconciliation_status,
    COUNT(*) AS record_count,
    COUNT(*) * 100.0 / (SELECT COUNT(*) FROM vw_holdings_price_validation) AS pct_of_total
FROM vw_holdings_price_validation
GROUP BY reconciliation_status
ORDER BY record_count DESC;

-- ============================================================
-- STEP 5: Top 20 outliers (greatest variance)
-- ============================================================

SELECT
    policyholder_number,
    security_code,
    asset_name,
    holding,
    local_bid_price,
    bid_value_local,
    reconciliation_pct,
    variance_from_100_pct,
    currency_code,
    include_flag,
    isin,
    sedol
FROM vw_holdings_price_validation
WHERE reconciliation_status != 'IN_RANGE'
ORDER BY ABS(variance_from_100_pct) DESC
LIMIT 20;

-- ============================================================
-- STEP 6: Breakdown by currency
-- ============================================================

SELECT
    currency_code,
    COUNT(*) AS record_count,
    SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) AS in_range_count,
    ROUND(100.0 * SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_in_range,
    MIN(reconciliation_pct) AS min_pct,
    MAX(reconciliation_pct) AS max_pct,
    ROUND(AVG(reconciliation_pct), 2) AS avg_pct
FROM vw_holdings_price_validation
GROUP BY currency_code
ORDER BY record_count DESC;

-- ============================================================
-- STEP 7: Breakdown by include/exclude flag
-- ============================================================

SELECT
    include_flag,
    COUNT(*) AS record_count,
    SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) AS in_range_count,
    ROUND(100.0 * SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_in_range
FROM vw_holdings_price_validation
GROUP BY include_flag
ORDER BY include_flag;


StatementMeta(, 62315e6e-5a1a-4655-95d5-11590f5b401b, 21, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1000 rows and 11 fields>

<Spark SQL result set with 1 rows and 7 fields>

<Spark SQL result set with 3 rows and 3 fields>

<Spark SQL result set with 20 rows and 12 fields>

<Spark SQL result set with 11 rows and 7 fields>

<Spark SQL result set with 2 rows and 4 fields>

In [5]:
%%sql
SELECT COUNT(*) AS stage2_rows
FROM individual_dfm_consolidated;

SELECT COUNT(*) AS validation_rows
FROM holdings_price_validation;

SHOW TABLES LIKE 'holdings_price_validation';

StatementMeta(, 62315e6e-5a1a-4655-95d5-11590f5b401b, 23, Finished, , Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `holdings_price_validation` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 4 pos 5;
'Aggregate [count(1) AS validation_rows#3232L]
+- 'UnresolvedRelation [holdings_price_validation], [], false


In [7]:
%%sql
-- ============================================================
-- Reconciliation Validation Setup (Fabric Spark SQL)
-- Creates BOTH a physical table and a view so they are easy to find
-- in OneLake Catalog / Power BI navigator.
-- ============================================================

-- 0) Context checks
SELECT current_database() AS active_database;
SELECT COUNT(*) AS source_row_count FROM individual_dfm_consolidated;

-- 1) Remove prior objects (safe idempotent)
DROP VIEW IF EXISTS vw_holdings_price_validation;
DROP TABLE IF EXISTS holdings_price_validation;

-- 2) Create physical Delta table for reliable catalog visibility
CREATE TABLE holdings_price_validation
USING DELTA
AS
WITH base AS (
    SELECT
        source_row_id,
        policyholder_number,
        security_code,
        asset_name,
        local_currency AS currency_code,
        include_flag,
        holding,
        local_bid_price,
        bid_value_gbp,
        CASE
            WHEN bid_value_gbp IS NULL OR fx_rate IS NULL OR fx_rate = 0 THEN NULL
            ELSE CAST(bid_value_gbp AS DOUBLE) / CAST(fx_rate AS DOUBLE)
        END AS bid_value_local,
        period,
        run_id,
        id_type,
        isin,
        sedol,
        other_security_id,
        accrued_interest_gbp,
        cash_value_gbp,
        fx_rate,
        exclusion_reason_code,
        identifier_chosen
    FROM individual_dfm_consolidated
    WHERE security_code IS NOT NULL
      AND TRIM(security_code) != ''
)
SELECT
    source_row_id,
    policyholder_number,
    security_code,
    asset_name,
    currency_code,
    include_flag,
    holding,
    local_bid_price,
    bid_value_local,
    bid_value_gbp,
    CASE
        WHEN holding IS NULL OR local_bid_price IS NULL OR bid_value_local IS NULL THEN NULL
        WHEN bid_value_local = 0 THEN NULL
        ELSE ROUND((CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0, 2)
    END AS reconciliation_pct,
    CASE
        WHEN holding IS NULL OR local_bid_price IS NULL OR bid_value_local IS NULL THEN 'MISSING_DATA'
        WHEN bid_value_local = 0 THEN 'ZERO_VALUE'
        WHEN (CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0 < 98.0 THEN 'BELOW_RANGE'
        WHEN (CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0 > 102.0 THEN 'ABOVE_RANGE'
        ELSE 'IN_RANGE'
    END AS reconciliation_status,
    CASE
        WHEN holding IS NULL OR local_bid_price IS NULL OR bid_value_local IS NULL THEN NULL
        WHEN bid_value_local = 0 THEN NULL
        ELSE ROUND(((CAST(holding AS DOUBLE) * CAST(local_bid_price AS DOUBLE)) / bid_value_local * 100.0) - 100.0, 2)
    END AS variance_from_100_pct,
    CASE WHEN holding IS NULL THEN 1 ELSE 0 END AS is_holding_null,
    CASE WHEN local_bid_price IS NULL THEN 1 ELSE 0 END AS is_price_null,
    CASE WHEN bid_value_local IS NULL THEN 1 ELSE 0 END AS is_bid_value_null,
    period,
    run_id,
    id_type,
    isin,
    sedol,
    other_security_id,
    accrued_interest_gbp,
    cash_value_gbp,
    fx_rate,
    exclusion_reason_code,
    identifier_chosen,
    current_timestamp() AS view_refresh_time
FROM base;

-- 3) Create a companion view (some tools prefer views)
CREATE VIEW vw_holdings_price_validation AS
SELECT *
FROM holdings_price_validation;

-- 4) Discovery checks (these prove object creation in current context)
SHOW TABLES LIKE 'holdings_price_validation';
SHOW TABLES LIKE 'vw_holdings_price_validation';

SELECT COUNT(*) AS validation_row_count
FROM holdings_price_validation;

-- 5) Quick preview
SELECT
    policyholder_number,
    security_code,
    asset_name,
    holding,
    local_bid_price,
    bid_value_local,
    reconciliation_pct,
    reconciliation_status,
    variance_from_100_pct,
    currency_code,
    include_flag
FROM holdings_price_validation
ORDER BY ABS(variance_from_100_pct) DESC
LIMIT 1000;

-- 6) Summary stats
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) AS records_in_range,
    SUM(CASE WHEN reconciliation_status != 'IN_RANGE' THEN 1 ELSE 0 END) AS outlier_records,
    ROUND(100.0 * SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_in_range,
    MIN(reconciliation_pct) AS min_reconciliation_pct,
    MAX(reconciliation_pct) AS max_reconciliation_pct,
    ROUND(AVG(reconciliation_pct), 2) AS avg_reconciliation_pct
FROM holdings_price_validation;

-- 7) Breakdown by status
SELECT
    reconciliation_status,
    COUNT(*) AS record_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM holdings_price_validation), 2) AS pct_of_total
FROM holdings_price_validation
GROUP BY reconciliation_status
ORDER BY record_count DESC;

-- 8) Top 20 outliers
SELECT
    policyholder_number,
    security_code,
    asset_name,
    holding,
    local_bid_price,
    bid_value_local,
    reconciliation_pct,
    variance_from_100_pct,
    currency_code,
    include_flag,
    isin,
    sedol
FROM holdings_price_validation
WHERE reconciliation_status != 'IN_RANGE'
ORDER BY ABS(variance_from_100_pct) DESC
LIMIT 20;

-- 9) Breakdown by currency
SELECT
    currency_code,
    COUNT(*) AS record_count,
    SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) AS in_range_count,
    ROUND(100.0 * SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_in_range,
    MIN(reconciliation_pct) AS min_pct,
    MAX(reconciliation_pct) AS max_pct,
    ROUND(AVG(reconciliation_pct), 2) AS avg_pct
FROM holdings_price_validation
GROUP BY currency_code
ORDER BY record_count DESC;

-- 10) Breakdown by include/exclude
SELECT
    include_flag,
    COUNT(*) AS record_count,
    SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) AS in_range_count,
    ROUND(100.0 * SUM(CASE WHEN reconciliation_status = 'IN_RANGE' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_in_range
FROM holdings_price_validation
GROUP BY include_flag
ORDER BY include_flag;


StatementMeta(, 62315e6e-5a1a-4655-95d5-11590f5b401b, 43, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

...

<Spark SQL result set with 1 rows and 7 fields>

<Spark SQL result set with 3 rows and 3 fields>

<Spark SQL result set with 20 rows and 12 fields>

<Spark SQL result set with 11 rows and 7 fields>

<Spark SQL result set with 2 rows and 4 fields>